1) The Problem: Internal Covariate Shift

    As a neural network trains, the weights of the earlier layers change. This means the distribution of the inputs to the deeper layers is constantly shifting. Ioffe and Szegedy called this Internal Covariate Shift. The deeper layers are constantly trying to adapt to a moving target, which forces us to use incredibly small learning rates and makes training painfully slow.
    
    BatchNorm solves this by forcibly anchoring the distribution of the inputs at every layer so they always have a consistent mean and variance.

2) Cricial step of Batch Normalization - Scale and Shift (Crucial Step)
    If we only normalized the data to mean 0 and variance 1, we might restrict the neural network's representational power (e.g., forcing all inputs to the linear regime of a Sigmoid function). To fix this, BatchNorm introduces two learnable parameters:$\gamma$ (Gamma): Scales the normalized value.$\beta$ (Beta): Shifts the normalized value.$$y_i = \gamma \hat{x}_i + \beta$$
    
    The network learns $\gamma$ and $\beta$ during backpropagation just like regular weights. If the network decides that the optimal distribution for this layer is not a mean of 0 and variance of 1, it can adjust $\gamma$ and $\beta$ to recover the original distribution.
    
3) The Catch: Training vs. InferenceThe detailed way to "do" BatchNorm changes depending on whether you are training or testing your model.

    - During Training: The mean and variance are calculated dynamically on the current mini-batch (as shown above).
    - During Inference (Testing/Production): You might be predicting on a single image or text input (batch size of 1). You can't calculate a batch mean on a single item. Therefore, during training, BatchNorm keeps a running moving average (Exponential Moving Average or EMA) of all the batch means and variances it sees. During inference, it uses these frozen historical statistics to normalize the new data.

4. BatchNorm vs. LayerNorm: The Dimensional Difference

    While BatchNorm was revolutionary, it struggles with small batch sizes (statistics become noisy) and Recurrent Neural Networks (RNNs). This led to the development of Layer Normalization (LayerNorm).
    
    The math (Steps 1-4) is almost exactly the same, but the axis of calculation changes:
    
    - Batch Normalization: Calculates mean and variance down the Batch dimension. It looks at one feature across all examples in the batch.
    - Layer Normalization: Calculates mean and variance across the Feature dimension. It looks at all features for one single example at a time. It completely ignores the batch size. (This is why LayerNorm is the standard in Transformers and LLMs).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader

# 1. Prepare the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 2. Split into Train and Validation
train_dataset, val_dataset = random_split(full_train_dataset, [0.8, 0.2])

# 3. Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [2]:
import torch
import torch.nn as nn

class BatchNormCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)   # 28x28 -> 14x14
        self.fc = nn.Linear(32 * 14 * 14, 10)

    def forward(self, x):
        x = self.conv(x)        # [B, 32, 28, 28]
        x = self.bn(x)
        x = self.relu(x)
        x = self.pool(x)        # [B, 32, 14, 14]
        x = x.view(x.size(0), -1)
        x = self.fc(x)          # [B, 10]
        return x

In [7]:
class LayerNormCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.ln = nn.LayerNorm([32, 28, 28])
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)   # 28
        self.fc = nn.Linear(32 * 14 * 14, 10)
    
    def forward(self, x):
        x = self.conv(x)        # [B, 32, 28, 28]
        x = self.ln(x)
        x = self.relu(x)
        x = self.pool(x)        # [B, 32, 14, 14]
        x = x.view(x.size(0), -1)
        x = self.fc(x)          # [B, 10]
        return x

In [9]:
#model = BatchNormCNN()
model = LayerNormCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

LayerNormCNN(
  (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (ln): LayerNorm((32, 28, 28), eps=1e-05, elementwise_affine=True)
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=6272, out_features=10, bias=True)
)

In [10]:
num_epoch = 5

for epoch in range(num_epoch):
    # Training phase
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
    
    train_loss /= len(train_loader.dataset)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted = torch.max(outputs.data, 1)
            correct += (predicted == labels).sum().item()
    
    val_loss /= len(val_loader.dataset)
    val_accuracy = correct / len(val_loader.dataset)
    
    print(f'Epoch [{epoch+1}/{num_epoch}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

Epoch [1/5], Train Loss: 0.1991, Val Loss: 0.1047, Val Accuracy: 0.9680
Epoch [2/5], Train Loss: 0.0801, Val Loss: 0.0853, Val Accuracy: 0.9743
Epoch [3/5], Train Loss: 0.0596, Val Loss: 0.0735, Val Accuracy: 0.9775
Epoch [4/5], Train Loss: 0.0508, Val Loss: 0.0661, Val Accuracy: 0.9815
Epoch [5/5], Train Loss: 0.0430, Val Loss: 0.0644, Val Accuracy: 0.9819
